# 奖励率 k 怎么算

企业两个决定：**自己做还是蒸馏**（innovate / imitate）、**公开还是不公开**（openness / secrecy）。
四格、一个奖励率 k。下面是 k 的算法。

术语不用自造：openness / secrecy 与「学习外溢 vs 竞争」见 [Dasaratha, *Innovation and Strategic Network Formation*](https://arxiv.org/abs/1911.06872)；奖励公开这一工具属于 [Wright (1983)](https://www.merit.unu.edu/publications/rmpdf/1994/rm1994-017.pdf) 的 **prize**（奖金制，与 patent、research contract 并列）；蒸馏与从头训练的算力对比见 [Rethinking LLM distillation under a fixed compute budget](https://aclanthology.org/2024.insights-1.6/)。

In [1]:
import sympy as sp

Cd, Cn = sp.symbols('C_d C_n', positive=True)   # 蒸馏成本 / 从头训练成本
L       = sp.symbols('L', positive=True)        # 公开的竞争损失 (competition)
Ed, En  = sp.symbols('E_d E_n', positive=True)  # 蒸馏者 / 原创者的训练额度
k, B    = sp.symbols('k B', positive=True)      # 奖励率 / 预算

print('假设： C_n > C_d ， E_n > E_d')

假设： C_n > C_d ， E_n > E_d


## 四格收益（净）

In [2]:
table = {
    ('imitate', 'secrecy'):  -Cd,
    ('imitate', 'openness'): -Cd + k*Ed - L,
    ('innovate','secrecy'):  -Cn,
    ('innovate','openness'): -Cn + k*En - L,
}
for (a, b), v in table.items():
    print(f'{a:9} / {b:9} = {v}')

imitate   / secrecy   = -C_d
imitate   / openness  = -C_d + E_d*k - L
innovate  / secrecy   = -C_n
innovate  / openness  = -C_n + E_n*k - L


## 边界 1：公开要划得来
公开的收益必须盖过竞争损失 L。

In [3]:
k1_im = sp.solve(sp.Eq(table[('imitate','openness')],  table[('imitate','secrecy')]),  k)[0]
k1_in = sp.solve(sp.Eq(table[('innovate','openness')], table[('innovate','secrecy')]), k)[0]
print('蒸馏者公开  > 蒸馏者闭源  =>  k >', k1_im)
print('原创者公开  > 原创者闭源  =>  k >', k1_in)
print()
print('E_n > E_d  =>  L/E_n < L/E_d ：原创者更容易愿意公开，蒸馏者要更高的 k。')

蒸馏者公开  > 蒸馏者闭源  =>  k > L/E_d
原创者公开  > 原创者闭源  =>  k > L/E_n

E_n > E_d  =>  L/E_n < L/E_d ：原创者更容易愿意公开，蒸馏者要更高的 k。


## 边界 2：还要有人愿意自己做
奖励**按各自的训练额度 E 算**，比较原创与蒸馏。

In [4]:
k2 = sp.solve(sp.Eq(table[('innovate','openness')], table[('imitate','openness')]), k)[0]
k2_nice = (Cn - Cd) / (En - Ed)          # 分子分母同乘 -1 的直观形式
assert sp.simplify(k2 - k2_nice) == 0    # 两种写法等价
print('自己做 > 蒸馏  =>  k >', k2_nice)
print()
print('分子 C_n - C_d ：原创多花的钱')
print('分母 E_n - E_d ：原创多训的额度')

自己做 > 蒸馏  =>  k > (-C_d + C_n)/(-E_d + E_n)

分子 C_n - C_d ：原创多花的钱
分母 E_n - E_d ：原创多训的额度


## 反例：奖励定额就无解
如果不按额度算（E_d = E_n），边界 2 消失。

In [5]:
denom = sp.simplify((En - Ed).subs(En, Ed))
print('E_n = E_d 时分母 =', denom, ' => 边界 2 无解')
print()
gap = sp.simplify(table[('innovate','openness')].subs(En, Ed)
                  - table[('imitate','openness')].subs(En, Ed))
print('自己做 - 蒸馏 =', gap, '（= C_d - C_n < 0）')
print('=> 无论 k 多大都没人自己做。奖励必须按训练额度算。')

E_n = E_d 时分母 = 0  => 边界 2 无解

自己做 - 蒸馏 = C_d - C_n （= C_d - C_n < 0）
=> 无论 k 多大都没人自己做。奖励必须按训练额度算。


## 预算上限
政府出钱，总预算 B。

In [6]:
print('k * (E_d + E_n) <= B   =>   k <=', B/(Ed + En))

k * (E_d + E_n) <= B   =>   k <= B/(E_d + E_n)


## k 的算法

In [7]:
def k_range(C_d, C_n, L, E_d, E_n, budget):
    """返回 (下限, 上限, 区间是否非空)。"""
    lower = max(L / E_d, (C_n - C_d) / (E_n - E_d))
    upper = budget / (E_d + E_n)
    return lower, upper, lower < upper

print('下限 = max( L/E_d , (C_n-C_d)/(E_n-E_d) )')
print('上限 = B/(E_d+E_n)')
print('非空 => 下限 < 上限')

下限 = max( L/E_d , (C_n-C_d)/(E_n-E_d) )
上限 = B/(E_d+E_n)
非空 => 下限 < 上限


## 三种市场状态

| k | 结果 |
|---|---|
| k < 下限 | 没人公开，全闭源（secrecy）：垄断高价，静态损失最大 |
| 下限 ≤ k ≤ 上限 | 有人自己做、有人公开（openness + innovation）：健康 |
| 奖励不按额度算 | 全在互相蒸馏（imitation）：无人自己做，动态损失最大 |

## 要算出具体 k，需要四个量

| 量 | 含义 | 候选来源 |
|---|---|---|
| C_d, C_n | 蒸馏成本、从头训练成本 | [固定算力预算下的蒸馏 vs 从头训练](https://aclanthology.org/2024.insights-1.6/) |
| L | 公开的竞争损失 | 闭源与开源模型的价格与收入差 |
| E_d, E_n | 训练额度（可核查） | 用电量或算力台账 |
| B | 预算 | 政策文件 |

**本 notebook 不估这些数**，只给算法。